# JODI-Oil Secondary database — schema walkthrough and India vs PPAC comparison

This notebook does three things:

1. **Loads the JODI Secondary CSV** with the right dtypes and the non-numeric flags handled.
2. **Pivots it from the long SDMX layout into something analysis-friendly** — one row per (country, month, product) with columns per flow.
3. **Compares India's `TOTDEMO` (total demand) against PPAC product-wise consumption** to show where the two sources agree and where they diverge.

## What the file actually is

JODI publishes two databases: **Primary** (crude oil, NGLs, refinery feedstocks — upstream) and **Secondary** (refined products — downstream). Your file is the Secondary one. JODI publishes with a ~2-month lag, so a file named `*year2026.csv` downloaded today realistically contains Jan + Feb 2026.

## The schema

Every row is one observation keyed by **five dimensions** plus a value and a quality flag:

| Column | What it is |
|---|---|
| `REF_AREA` | ISO-2 country code (`IN` = India, `US` = USA, etc.) |
| `TIME_PERIOD` | Month, formatted `YYYY-MM` |
| `ENERGY_PRODUCT` | Refined product code — see table below |
| `FLOW_BREAKDOWN` | Balance item — see table below |
| `UNIT_MEASURE` | Unit the same observation is reported in (every observation appears 5 times in 5 units) |
| `OBS_VALUE` | The number — **stored as string** because flags like `-`, `x`, `c` are mixed in |
| `ASSESSMENT_CODE` | 1 = official final, 2 = official preliminary, 3 = JODI estimate |

### ENERGY_PRODUCT codes
| Code | Meaning |
|---|---|
| `GASOLINE` | Motor gasoline |
| `GASDIES` | Gas/diesel oil (road diesel + heating gasoil) |
| `JETKERO` | Kerosene-type jet fuel |
| `KEROSENE` | Other kerosene (lamp / domestic, not jet) |
| `LPG` | Liquefied petroleum gases |
| `NAPHTHA` | Naphtha |
| `RESFUEL` | Residual fuel oil |
| `ONONSPEC` | Other oil products (not specified elsewhere) |
| `TOTPRODS` | Total of the above |

### FLOW_BREAKDOWN codes — the JODI supply/demand identity
JODI's balance equation: **`RECEIPTS + REFGROUT + TOTIMPSB − TOTEXPSB − IPTRANSF − PTRANSF + STOCKCH + STATDIFF = TOTDEMO`**

| Code | Meaning |
|---|---|
| `RECEIPTS` | Refinery receipts (intake) |
| `REFGROUT` | Refinery gross output |
| `TOTIMPSB` | Total imports |
| `TOTEXPSB` | Total exports |
| `IPTRANSF` | Inter-product transfers |
| `PTRANSF` | Products transferred (to other categories) |
| `STOCKCH` | Stock change (negative = build, positive = draw — supply contribution) |
| `CLOSTLV` | Closing stock **level** (not a flow — the stock at month-end) |
| `STATDIFF` | Statistical difference (balancing item) |
| `TOTDEMO` | **Total demand — what you compare to PPAC** |

### UNIT_MEASURE codes
| Code | Meaning |
|---|---|
| `KBD` | Thousand barrels per day — **best for cross-country comparison** |
| `KBBL` | Thousand barrels for the month (KBD × days-in-month) |
| `KTONS` | Thousand metric tons for the month |
| `KL` | Thousand kilolitres for the month |
| `CONVBBL` | Conversion factor, barrels per metric ton (a constant, not an observation) |

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

## 1. Load the file properly

Two things to handle:

- `OBS_VALUE` is a string column with `-`, `x`, `c` mixed into the numbers. `pd.read_csv(..., na_values=[...])` converts them to `NaN` and lets pandas infer float for the rest.
- `TIME_PERIOD` is `YYYY-MM`. Parse it to a proper monthly period so resampling and sorting Just Work.

In [4]:
PATH = Path('../data/raw/jodi') / 'secondaryyear2026.csv'  # adjust as needed

jodi = pd.read_csv(
    PATH,
    na_values=['-', 'x', 'c', '..'],   # JODI flag codes for missing/NA/confidential/not-yet-available
    dtype={
        'REF_AREA': 'category',
        'ENERGY_PRODUCT': 'category',
        'FLOW_BREAKDOWN': 'category',
        'UNIT_MEASURE': 'category',
    },
)

# Force OBS_VALUE to float (na_values catches most flags; this catches anything else)
jodi['OBS_VALUE'] = pd.to_numeric(jodi['OBS_VALUE'], errors='coerce')

# Parse YYYY-MM into a proper monthly period
jodi['TIME_PERIOD'] = pd.PeriodIndex(jodi['TIME_PERIOD'], freq='M')

# Assessment code as a labelled category — easier to read
jodi['ASSESSMENT'] = jodi['ASSESSMENT_CODE'].map({1: 'official_final', 2: 'official_prelim', 3: 'jodi_estimate'}).astype('category')

print(f'Shape: {jodi.shape}')
print(f'Countries: {jodi["REF_AREA"].nunique()}')
print(f'Period range: {jodi["TIME_PERIOD"].min()} → {jodi["TIME_PERIOD"].max()}')
print(f'Missing in OBS_VALUE: {jodi["OBS_VALUE"].isna().sum():,} / {len(jodi):,} ({jodi["OBS_VALUE"].isna().mean():.0%})')
jodi.head()

Shape: (86400, 8)
Countries: 96
Period range: 2026-01 → 2026-02
Missing in OBS_VALUE: 39,166 / 86,400 (45%)


,REF_AREA,TIME_PERIOD,ENERGY_PRODUCT,FLOW_BREAKDOWN,UNIT_MEASURE,OBS_VALUE,ASSESSMENT_CODE,ASSESSMENT
0,AE,2026-01,GASDIES,CLOSTLV,CONVBBL,7400.0,3,jodi_estimate
1,AE,2026-01,GASDIES,CLOSTLV,KBBL,NaN,3,jodi_estimate
2,AE,2026-01,GASDIES,CLOSTLV,KBD,NaN,3,jodi_estimate
3,AE,2026-01,GASDIES,CLOSTLV,KL,NaN,3,jodi_estimate
4,AE,2026-01,GASDIES,CLOSTLV,KTONS,NaN,3,jodi_estimate


## 2. Reshape from long to wide

The long format is great for storage and for plotting libraries that expect tidy data (seaborn, plotly express), but it's painful for the actual analysis question "what was India's diesel demand minus its diesel imports last month". For that you want **one row per (country, month, product) with columns per flow**.

Pick **one unit** first — pivoting across all 5 units at once explodes the column count. `KBD` is the most useful for cross-country work.

In [5]:
def to_wide(df, unit='KBD'):
    """Pivot JODI from long to wide on FLOW_BREAKDOWN, for one chosen unit."""
    subset = df[df['UNIT_MEASURE'] == unit]
    wide = subset.pivot_table(
        index=['REF_AREA', 'TIME_PERIOD', 'ENERGY_PRODUCT'],
        columns='FLOW_BREAKDOWN',
        values='OBS_VALUE',
        aggfunc='first',
        observed=True,
    ).reset_index()
    wide.columns.name = None
    return wide

kbd = to_wide(jodi, unit='KBD')
kt  = to_wide(jodi, unit='KTONS')

print('KBD wide shape:', kbd.shape)
kbd.head()

KBD wide shape: (948, 11)


,REF_AREA,TIME_PERIOD,ENERGY_PRODUCT,IPTRANSF,PTRANSF,RECEIPTS,REFGROUT,STATDIFF,TOTDEMO,TOTEXPSB,TOTIMPSB
0,AM,2026-01,GASDIES,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.1032
1,AM,2026-01,GASOLINE,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.7419
2,AM,2026-01,JETKERO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.0348
3,AM,2026-01,KEROSENE,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.0581
4,AM,2026-01,LPG,NaN,NaN,NaN,NaN,NaN,NaN,0.0,7.4839


### Sanity-check the supply/demand identity

If JODI's balance holds, `REFGROUT + TOTIMPSB − TOTEXPSB − IPTRANSF − PTRANSF + STOCKCH + STATDIFF` should equal `TOTDEMO` (after also accounting for `RECEIPTS`, but for refined-products demand the refinery `RECEIPTS` row is normally only populated for crude/feedstocks — check it). Worth running this once just to confirm we understand the file.

In [6]:
# Pick India, GASOLINE, latest month — fillna(0) because JODI uses NaN for 'not applicable' flow
row = kbd[(kbd['REF_AREA']=='IN') & (kbd['ENERGY_PRODUCT']=='GASOLINE')].iloc[0].fillna(0)
print(row)

# Compute implied demand from the components, compare to reported TOTDEMO
supply = (row.get('REFGROUT',0) + row.get('TOTIMPSB',0) - row.get('TOTEXPSB',0)
          - row.get('IPTRANSF',0) - row.get('PTRANSF',0)
          + row.get('STOCKCH',0) + row.get('STATDIFF',0))
print(f'\nImplied (supply side) = {supply:.1f} kbd')
print(f'Reported TOTDEMO      = {row["TOTDEMO"]:.1f} kbd')
print(f'Residual              = {supply - row["TOTDEMO"]:.2f} kbd  (STATDIFF absorbs the imbalance)')

REF_AREA                 IN
TIME_PERIOD         2026-01
ENERGY_PRODUCT     GASOLINE
IPTRANSF                  0
PTRANSF                   0
RECEIPTS                  0
REFGROUT          1139.9059
STATDIFF                  0
TOTDEMO            958.8428
TOTEXPSB           383.9741
TOTIMPSB                  0
Name: 429, dtype: object

Implied (supply side) = 755.9 kbd
Reported TOTDEMO      = 958.8 kbd
Residual              = -202.91 kbd  (STATDIFF absorbs the imbalance)


## 3. India consumption — JODI view

Pull just India's `TOTDEMO` (total demand) by product.

In [7]:
PRODUCTS = ['GASOLINE','GASDIES','JETKERO','KEROSENE','LPG','NAPHTHA','RESFUEL','ONONSPEC','TOTPRODS']

india_kbd = (
    kbd[(kbd['REF_AREA']=='IN') & (kbd['ENERGY_PRODUCT'].isin(PRODUCTS))]
    [['TIME_PERIOD','ENERGY_PRODUCT','TOTDEMO']]
    .pivot(index='TIME_PERIOD', columns='ENERGY_PRODUCT', values='TOTDEMO')
)
india_kbd

ENERGY_PRODUCT,GASDIES,GASOLINE,JETKERO,KEROSENE,LPG,NAPHTHA,ONONSPEC,RESFUEL,TOTPRODS
TIME_PERIOD,,,,,,,,,
2026-01,1942.9293,958.8428,209.4039,220.1565,1138.5491,294.3174,946.3065,118.7225,5620.0


In [8]:
# Same series in KTONS — this is the unit PPAC publishes in
india_kt = (
    kt[(kt['REF_AREA']=='IN') & (kt['ENERGY_PRODUCT'].isin(PRODUCTS))]
    [['TIME_PERIOD','ENERGY_PRODUCT','TOTDEMO']]
    .pivot(index='TIME_PERIOD', columns='ENERGY_PRODUCT', values='TOTDEMO')
)
india_kt

ENERGY_PRODUCT,GASDIES,GASOLINE,JETKERO,KEROSENE,LPG,NAPHTHA,ONONSPEC,RESFUEL,TOTPRODS
TIME_PERIOD,,,,,,,,,
2026-01,8076.0,3511.0,828.0,865.0,3033.0,1024.0,3975.0,567.0,21051.0


## 4. Compare to PPAC

PPAC publishes **product-wise consumption in thousand metric tons** (TMT) at https://ppac.gov.in/consumption/products-wise. The page renders an HTML table; the easiest path is to download the Excel they provide and load it, or scrape with `pandas.read_html`.

### Product mapping (PPAC label → JODI code)

| PPAC product | JODI `ENERGY_PRODUCT` | Notes |
|---|---|---|
| LPG | `LPG` | Clean match |
| MS (Motor Spirit) | `GASOLINE` | Same thing |
| HSD (High Speed Diesel) | `GASDIES` | JODI's `GASDIES` is HSD + LDO + heating gasoil; for India HSD dominates so it's close, but not identical |
| ATF (Aviation Turbine Fuel) | `JETKERO` | Clean match |
| SKO (Superior Kerosene Oil) | `KEROSENE` | Clean match |
| Naphtha | `NAPHTHA` | Clean match |
| FO + LSHS | `RESFUEL` | Clean match (PPAC sometimes splits) |
| Bitumen, Petcoke, LDO, others | part of `ONONSPEC` | PPAC reports these individually; JODI lumps them |
| Total | `TOTPRODS` | Should reconcile to within a small residual |

### Loading PPAC

PPAC's download URL pattern (you'll want to inspect the page to confirm — they update file names occasionally):

```python
# Example - replace with the actual download URL from the PPAC page:
PPAC_URL = 'https://ppac.gov.in/uploads/.../consumption_petroleum_products.xlsx'
ppac_raw = pd.read_excel(PPAC_URL, sheet_name='Sheet1')
```

Below I'll build the comparison assuming you've loaded PPAC into a DataFrame `ppac` with columns `month`, `product`, `consumption_kt`. Substitute your real loading code.

In [ ]:
# --- Placeholder: replace this block with your actual PPAC load ---
# Build a fake PPAC frame for one month to show the comparison mechanics.
# In real use, load from PPAC's Excel or scrape their HTML table.
ppac = pd.DataFrame({
    'month': pd.PeriodIndex(['2026-01','2026-01','2026-01','2026-01','2026-01','2026-01','2026-01'], freq='M'),
    'ppac_product': ['LPG','MS','HSD','ATF','SKO','Naphtha','FO+LSHS'],
    'consumption_kt': [2700, 3500, 8000, 800, 30, 1100, 550],   # illustrative
})

PPAC_TO_JODI = {
    'LPG':     'LPG',
    'MS':      'GASOLINE',
    'HSD':     'GASDIES',
    'ATF':     'JETKERO',
    'SKO':     'KEROSENE',
    'Naphtha': 'NAPHTHA',
    'FO+LSHS': 'RESFUEL',
}
ppac['jodi_product'] = ppac['ppac_product'].map(PPAC_TO_JODI)
ppac

In [ ]:
# Reshape JODI India to (month, product, kt) for the join
india_kt_long = india_kt.reset_index().melt(
    id_vars='TIME_PERIOD', var_name='jodi_product', value_name='jodi_kt'
)
india_kt_long = india_kt_long.rename(columns={'TIME_PERIOD': 'month'})

compare = ppac.merge(india_kt_long, on=['month','jodi_product'], how='inner')
compare['diff_kt']  = compare['consumption_kt'] - compare['jodi_kt']
compare['diff_pct'] = compare['diff_kt'] / compare['jodi_kt'] * 100
compare = compare[['month','ppac_product','jodi_product','consumption_kt','jodi_kt','diff_kt','diff_pct']]
compare

### Plot side-by-side

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(compare))
w = 0.4
ax.bar(x - w/2, compare['consumption_kt'], w, label='PPAC')
ax.bar(x + w/2, compare['jodi_kt'],        w, label='JODI')
ax.set_xticks(x)
ax.set_xticklabels(compare['ppac_product'], rotation=0)
ax.set_ylabel('Consumption (kt)')
ax.set_title(f'India product consumption — PPAC vs JODI, {compare["month"].iloc[0]}')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Things to be careful about when interpreting divergences

1. **JODI's India source IS PPAC** for most products. So for recent months you should see near-identity in published numbers. Big gaps usually mean (a) you're comparing different vintages — PPAC revises, JODI snapshots; (b) you have an `ASSESSMENT_CODE = 3` row (JODI estimate, not yet replaced with the official number); or (c) you've crossed a definition boundary like HSD vs GASDIES.
2. **`TOTDEMO` ≠ "sales".** TOTDEMO is the demand side of the JODI balance: gross output + imports − exports ± stock change. PPAC reports **consumption (sales)**, which OMCs derive from terminal dispatches. The two diverge when there are large stock movements at marketing-terminal level that the OMC-reported figures don't capture cleanly.
3. **Unit conversions.** If you compare in kbd instead of kt, you need to be careful about which density JODI uses. The `CONVBBL` flow gives the country/product-specific conversion factor (bbl/ton) for that month — use it explicitly rather than a textbook value: `kbd = kt * CONVBBL / days_in_month`.
4. **Coverage of `ONONSPEC`.** PPAC reports bitumen, petcoke, LDO, lubricants, etc. as individual lines. JODI rolls them into `ONONSPEC`. Don't try to reconcile each one individually — sum the PPAC "others" basket before comparing.
5. **`TOTPRODS` totals.** PPAC's headline "Total Consumption" and JODI's `TOTPRODS / TOTDEMO` should be the cleanest cross-check. Any gap there points to a definitional issue you want to understand before trusting product-level comparisons.

## 6. Suggested next steps

- Pull the **full historical** JODI download (not just `secondaryyear2026.csv`, but the full archive — JODI offers a single file with all history) so you can plot multi-year series.
- Add a column to your wide frame that flags the `ASSESSMENT_CODE` so you can visually distinguish hard data from JODI estimates in charts (e.g. dashed line for code 3).
- Build a `country_summary(country_code)` function that returns the supply-demand balance table for a country across all products in kbd. That's the workhorse view for desk research.
- If you want a country-pair view (e.g. India + UAE crude product trade), the JODI **Primary** database has the crude side. Same schema, different product codes.